In [5]:
import numpy as np
import torch
import time
from scipy import sparse as sp
from scipy.linalg import eigh
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

pauli_matrices = {
    1: np.array([[0, 1], [1, 0]], dtype=complex),
    2: np.array([[0, -1j], [1j, 0]], dtype=complex),
    3: np.array([[1, 0], [0, -1]], dtype=complex),
    0: np.eye(2, dtype=complex)
}
pauli_matrices_sparse = {key: sp.csr_matrix(value) for key, value in pauli_matrices.items()}

def pauli_site(x, i, L):
    """Generate Pauli matrix on site i (sparse)"""
    if L == 1:
        return pauli_matrices_sparse[x]
    if i == 1:
        return sp.kron(pauli_matrices_sparse[x], sp.eye(2**(L-1), format='csr'))
    else:
        return sp.kron(sp.eye(2), pauli_site(x, i-1, L-1), format='csr')

def pauli_int(spin, site, L):
    """Generate interaction term: product of Pauli matrices between site and site+1 (periodic boundary)"""
    dim = 2**L
    if site == L:
        op = sp.eye(dim, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == L or i == 1:
                op = op @ pauli_site(spin, i, L)
        return op
    else:
        op = sp.eye(dim, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == site or i == site+1:
                op = op @ pauli_site(spin, i, L)
        return op

def xxz_hamiltonian(L, delta=1):
    """Pure XXZ Hamiltonian (nearest neighbor)"""
    dim = 2**L
    H = sp.csr_matrix((dim, dim), dtype=complex)
    for site in range(1, L+1):
        H += pauli_int(1, site, L)   # XX
        H += pauli_int(2, site, L)   # YY
        H += delta * pauli_int(3, site, L)  # ZZ
    return H

def periodic_field_hamiltonian(L, delta=1, Q=np.pi, h=0.5):
    """XXZ + longitudinal periodic field: H = H_XXZ + h * Σ_n cos(Q n) σ^z_n"""
    H0 = xxz_hamiltonian(L, delta)
    dim = 2**L
    Hz = sp.csr_matrix((dim, dim), dtype=complex)
    for n in range(1, L+1):
        Hz += np.cos(Q * n) * pauli_site(3, n, L)
    H = H0 + h * Hz
    return H

def next_nearest_hamiltonian(L, delta=1, lambda_val=0.3):
    """XXZ + next‑nearest neighbor Heisenberg interaction: H = H_XXZ + λ * Σ_n (S_n·S_{n+2})"""
    H0 = xxz_hamiltonian(L, delta)
    dim = 2**L
    H1 = sp.csr_matrix((dim, dim), dtype=complex)
    for n in range(1, L+1):
        n2 = n + 2
        if n2 > L:
            n2 = n2 - L
        H1 += pauli_site(1, n, L) @ pauli_site(1, n2, L)
        H1 += pauli_site(2, n, L) @ pauli_site(2, n2, L)
        H1 += pauli_site(3, n, L) @ pauli_site(3, n2, L)
    H = H0 + lambda_val * H1
    return H

def build_hamiltonian(model, L, **kwargs):
    """Construct Hamiltonian according to the chosen model"""
    if model == "XXZ":
        delta = kwargs.get('delta', 1.0)
        return xxz_hamiltonian(L, delta)
    elif model == "PeriodicField":
        delta = kwargs.get('delta', 1.0)
        Q = kwargs.get('Q', np.pi)
        h = kwargs.get('h', 0.5)
        return periodic_field_hamiltonian(L, delta, Q, h)
    elif model == "NextNearest":
        delta = kwargs.get('delta', 1.0)
        lambda_val = kwargs.get('lambda_val', 0.3)
        return next_nearest_hamiltonian(L, delta, lambda_val)
    else:
        raise ValueError(f"Unknown model: {model}, available: XXZ, PeriodicField, NextNearest")

def local_ops(dtype=torch.complex64, device="cpu"):
    Sz = torch.tensor([[0.5, 0.0], [0.0, -0.5]], dtype=dtype, device=device)
    Sp = torch.tensor([[0.0, 1.0], [0.0, 0.0]], dtype=dtype, device=device)
    Sm = torch.tensor([[0.0, 0.0], [1.0, 0.0]], dtype=dtype, device=device)
    Sx = torch.tensor([[0.0, 1.0], [1.0, 0.0]], dtype=dtype, device=device)
    Sy = torch.tensor([[0.0, -1j], [1j, 0.0]], dtype=dtype, device=device)
    I2 = torch.eye(2, dtype=dtype, device=device)
    return I2, Sx, Sy, Sz, Sp, Sm

def precompute_perms(L: int):
    perms, inv_perms = [], []
    axes = list(range(L))
    for s in range(L):
        p = [s] + [a for a in axes if a != s]
        inv = np.argsort(p)
        perms.append(p)
        inv_perms.append(inv.tolist())
    return perms, inv_perms

def apply_one_site(op2, site, vec, L, perms, inv_perms):
    psi = vec.reshape([2]*L).permute(perms[site]).reshape(2, -1)
    psi2 = op2 @ psi
    out = psi2.reshape([2]*L).permute(inv_perms[site]).reshape(-1)
    return out

def B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms):
    dtype, device = psi.dtype, psi.device
    ii = torch.tensor(1j, dtype=dtype, device=device)
    V11 = psi.clone()
    V12 = torch.zeros_like(psi)
    V21 = torch.zeros_like(psi)
    V22 = psi.clone()
    for site in range(L):
        SzV11 = apply_one_site(Sz, site, V11, L, perms, inv_perms)
        SzV12 = apply_one_site(Sz, site, V12, L, perms, inv_perms)
        SzV21 = apply_one_site(Sz, site, V21, L, perms, inv_perms)
        SzV22 = apply_one_site(Sz, site, V22, L, perms, inv_perms)
        SmV21 = apply_one_site(Sm, site, V21, L, perms, inv_perms)
        SmV22 = apply_one_site(Sm, site, V22, L, perms, inv_perms)
        SpV11 = apply_one_site(Sp, site, V11, L, perms, inv_perms)
        SpV12 = apply_one_site(Sp, site, V12, L, perms, inv_perms)
        L11V11 = u * V11 + ii * SzV11
        L11V12 = u * V12 + ii * SzV12
        L22V21 = u * V21 - ii * SzV21
        L22V22 = u * V22 - ii * SzV22
        L12V21 = ii * SmV21
        L12V22 = ii * SmV22
        L21V11 = ii * SpV11
        L21V12 = ii * SpV12
        N11 = L11V11 + L12V21
        N12 = L11V12 + L12V22
        N21 = L21V11 + L22V21
        N22 = L21V12 + L22V22
        V11, V12, V21, V22 = N11, N12, N21, N22
    return V12

def vacuum_state(L, dtype=torch.complex64, device="cpu"):
    v = torch.zeros(2**L, dtype=dtype, device=device)
    v[0] = 1.0 + 0.0j
    return v

def bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True):
    psi = vacuum_state(L, dtype=I2.dtype, device=I2.device)
    for layer_params in params_list:
        u = layer_params[0] + 1j * layer_params[1]
        psi = B_action(u, psi, L, I2, Sz, Sp, Sm, perms, inv_perms)
    if normalize:
        norm = torch.sqrt(torch.real(torch.vdot(psi, psi)))
        if norm > 1e-15:
            psi = psi / norm
    return psi

def exact_diagonalization(H, num_states=3):
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    eigenvalues, eigenvectors = eigh(H_dense)
    results = []
    for i in range(min(num_states, len(eigenvalues))):
        results.append({'energy': eigenvalues[i], 'state': eigenvectors[:, i]})
    return results

def pytorch_loss_func(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params_per_layer = 2
    total_layers = len(params) // params_per_layer
    params_list = []
    for layer in range(total_layers):
        start_idx = layer * params_per_layer
        re_part = params[start_idx]
        im_part = params[start_idx + 1]
        params_list.append([re_part, im_part])
    psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    return energy / norm2

def torch_wrapper_loss(params_numpy, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)
    loss = pytorch_loss_func(params, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)
    loss.backward()
    loss_value = loss.item()
    grad_value = params.grad.numpy().astype(np.float64)
    return loss_value, grad_value

def optimize_ground_state(model, L, num_layers=3, num_attempts=5, maxiter=6000,
                          device="cpu", **model_kwargs):
    """
    Unified optimization interface with special handling for degenerate ground states.
    For the NextNearest model with lambda_val = 0.5, fidelity is computed as the sum of
    squared overlaps with the ground state and the first excited state (degenerate subspace).
    Otherwise, fidelity is simply the squared overlap with the exact ground state.
    """
    print(f"Model: {model}")
    print(f"Number of sites L = {L}")
    print(f"Bethe ansatz layers: {num_layers}")
    print(f"Device: {device}")
    if model == "XXZ":
        delta = model_kwargs.get('delta', 1.0)
        print(f"Anisotropy Δ = {delta}")
    elif model == "PeriodicField":
        delta = model_kwargs.get('delta', 1.0)
        Q = model_kwargs.get('Q', np.pi)
        h = model_kwargs.get('h', 0.5)
        print(f"Anisotropy Δ = {delta}, wave number Q = {Q:.4f}, field strength h = {h}")
    elif model == "NextNearest":
        delta = model_kwargs.get('delta', 1.0)
        lambda_val = model_kwargs.get('lambda_val', 0.3)
        print(f"Anisotropy Δ = {delta}, next‑nearest neighbor strength λ = {lambda_val}")

    H = build_hamiltonian(model, L, **model_kwargs)
    exact_states = exact_diagonalization(H, num_states=3)
    ground_energy = exact_states[0]['energy']
    excited1_energy = exact_states[1]['energy']
    print(f"Exact ground state energy: {ground_energy:.8f}")
    print(f"Exact first excited state energy: {excited1_energy:.8f}")

    degenerate_case = (model == "NextNearest" and 
                       abs(model_kwargs.get('lambda_val', 0.3) - 0.5) < 1e-12)
    if degenerate_case:
        if len(exact_states) < 2:
            raise ValueError("Not enough states to compute degenerate fidelity.")
        if abs(ground_energy - excited1_energy) > 1e-10:
            print(f"Warning: energies are not exactly degenerate: {ground_energy} vs {excited1_energy}")

    I2, Sx, Sy, Sz, Sp, Sm = local_ops(dtype=torch.complex128, device=device)
    perms, inv_perms = precompute_perms(L)
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    H_tensor = torch.tensor(H_dense, dtype=torch.complex128, device=device)
    ground_state_tensor = torch.tensor(exact_states[0]['state'], dtype=torch.complex128, device=device)
    if degenerate_case:
        excited1_state_tensor = torch.tensor(exact_states[1]['state'], dtype=torch.complex128, device=device)

    best_energy = float('inf')
    best_params = None
    best_psi = None
    best_fidelity = 0.0

    for attempt in range(num_attempts):
        print(f"\nAttempt {attempt+1}/{num_attempts}:")
        total_params = num_layers * 2
        x0 = np.zeros(total_params, dtype=np.float64)
        for layer in range(num_layers):
            x0[layer*2:layer*2+2] = np.random.normal(-5, 5, 2)

        def loss_wrapper(p):
            return torch_wrapper_loss(p, H_tensor, L, I2, Sz, Sp, Sm, perms, inv_perms, num_layers)

        result = minimize(
            lambda p: loss_wrapper(p)[0],
            x0,
            method='L-BFGS-B',
            jac=lambda p: loss_wrapper(p)[1],
            options={'maxiter': maxiter, 'ftol': 1e-12, 'gtol': 1e-10},
            bounds=[(-3, 3)] * total_params
        )
        final_params = result.x
        final_loss = result.fun

        params_list = []
        for layer in range(num_layers):
            re_part = final_params[layer*2]
            im_part = final_params[layer*2+1]
            params_list.append([re_part, im_part])
        psi = bethe_state_multilayer(params_list, L, I2, Sz, Sp, Sm, perms, inv_perms, normalize=True)
        psi_numpy = psi.cpu().detach().numpy()
        norm = np.vdot(psi_numpy, psi_numpy)

        if sp.issparse(H):
            H_psi = H.dot(psi_numpy)
        else:
            H_psi = H @ psi_numpy
        energy = np.real(np.vdot(psi_numpy, H_psi) / norm)

        if degenerate_case:
            overlap_gs = np.abs(np.vdot(psi_numpy, exact_states[0]['state']))**2 / norm
            overlap_ex1 = np.abs(np.vdot(psi_numpy, exact_states[1]['state']))**2 / norm
            fidelity = overlap_gs + overlap_ex1
            print(f"  Loss value: {final_loss:.8f}")
            print(f"  Energy expectation: {energy:.8f}")
            print(f"  Overlap^2 with ground state: {overlap_gs:.8f}")
            print(f"  Overlap^2 with first excited: {overlap_ex1:.8f}")
            print(f"  Total fidelity (sum): {fidelity:.8f}")
        else:
            fidelity = np.abs(np.vdot(psi_numpy, exact_states[0]['state']))**2 / norm
            print(f"  Loss value: {final_loss:.8f}")
            print(f"  Energy expectation: {energy:.8f}")
            print(f"  Ground state fidelity: {fidelity:.8f}")

        if energy < best_energy:
            best_energy = energy
            best_params = final_params
            best_psi = psi_numpy
            best_fidelity = fidelity

    return best_energy, best_params, best_psi, exact_states, best_fidelity, degenerate_case

if __name__ == "__main__":
    model = "NextNearest"   
    L = 4
    num_layers = L // 2
    num_attempts = 5
    device = "cuda" if torch.cuda.is_available() else "cpu"

    if model == "XXZ":
        model_kwargs = {'delta': 1.0}
    elif model == "PeriodicField":
        model_kwargs = {'delta': 1.0, 'Q': np.pi, 'h': 0.85}
    elif model == "NextNearest":
        model_kwargs = {'delta': 1.0, 'lambda_val': 0.5}
    else:
        raise ValueError("Unsupported model")

    start_time = time.time()
    best_energy, best_params, best_psi, exact_states, best_fidelity, degenerate = optimize_ground_state(
        model=model,
        L=L,
        num_layers=num_layers,
        num_attempts=num_attempts,
        maxiter=10000,
        device=device,
        **model_kwargs
    )
    elapsed_time = time.time() - start_time

    if best_params is not None:
        print(f"Bethe ansatz ground state energy: {best_energy:.8f}")
        print(f"Exact ground state energy:          {exact_states[0]['energy']:.8f}")
        print(f"Absolute energy error:          {abs(best_energy - exact_states[0]['energy']):.8f}")
        if degenerate:
            print(f"Total fidelity (sum over degenerate subspace): {best_fidelity:.8f}")
            overlap_gs = np.abs(np.vdot(best_psi, exact_states[0]['state']))**2
            overlap_ex1 = np.abs(np.vdot(best_psi, exact_states[1]['state']))**2
            print(f"Overlap^2 with exact ground state:      {overlap_gs:.8f}")
            print(f"Overlap^2 with first excited state:     {overlap_ex1:.8f}")
        else:
            print(f"Ground state fidelity:            {best_fidelity:.8f}")
        for i in range(1, min(3, len(exact_states))):
            overlap = np.abs(np.vdot(best_psi, exact_states[i]['state']))
            print(f"Overlap with {i}th excited state:      {overlap:.8f}")

        print("\nOptimized Bethe parameters u (per layer):")
        for layer in range(num_layers):
            re = best_params[layer*2]
            im = best_params[layer*2+1]
            print(f"  Layer {layer+1}: {re:.6f} + {im:.6f}i")
        print(f"\nOptimization time: {elapsed_time:.2f} seconds")

Model: NextNearest
Number of sites L = 4
Bethe ansatz layers: 2
Device: cpu
Anisotropy Δ = 1.0, next‑nearest neighbor strength λ = 0.5
Exact ground state energy: -6.00000000
Exact first excited state energy: -6.00000000

Attempt 1/5:
  Loss value: -5.13043478
  Energy expectation: -5.13043478
  Overlap^2 with ground state: 0.78260870+0.00000000j
  Overlap^2 with first excited: 0.02898551+0.00000000j
  Total fidelity (sum): 0.81159420+0.00000000j

Attempt 2/5:
  Loss value: -6.00000000
  Energy expectation: -6.00000000
  Overlap^2 with ground state: 0.75000000+0.00000000j
  Overlap^2 with first excited: 0.25000000+0.00000000j
  Total fidelity (sum): 1.00000000+0.00000000j

Attempt 3/5:
  Loss value: -5.99852719
  Energy expectation: -5.99852719
  Overlap^2 with ground state: 0.24999415+0.00000000j
  Overlap^2 with first excited: 0.74963766+0.00000000j
  Total fidelity (sum): 0.99963181+0.00000000j

Attempt 4/5:
  Loss value: -6.00000000
  Energy expectation: -6.00000000
  Overlap^2 with